# FMM geometry and interpolation

Purpose: reusable, differentiable FMM foundations. Inputs: function arguments; outputs: definitions only. Expected runtime: seconds for import. Checkpoints: none (stateless module). Resume: restart kernel and execute all cells; downstream notebooks load only `ahr-export` cells.


In [ ]:
from pathlib import Path
import json, os
PROJECT_ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).expanduser().resolve()
_bootstrap = json.loads((PROJECT_ROOT / "fmm_jepa/bootstrap.ipynb").read_text())
_bootstrap_namespace = {}
for _cell in _bootstrap["cells"]:
    if _cell["cell_type"] == "code" and "ahr-export" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), "bootstrap.ipynb#ahr-export", "exec"), _bootstrap_namespace)
_bootstrap_namespace["install_notebook_package"](PROJECT_ROOT)


In [ ]:
"""Uniform padded quadtree and tensor-product Chebyshev transfers."""
from dataclasses import dataclass
from functools import lru_cache
import math
import torch


def patch_coordinates(grid, device=None, dtype=torch.float32):
    rows, cols = map(int, grid)
    if rows < 1 or cols < 1:
        raise ValueError("grid dimensions must be positive")
    yy, xx = torch.meshgrid(torch.arange(rows, device=device, dtype=dtype), torch.arange(cols, device=device, dtype=dtype), indexing="ij")
    return torch.stack(((xx.reshape(-1)+0.5)/cols, (yy.reshape(-1)+0.5)/rows), dim=-1)


def chebyshev_nodes(order, device=None, dtype=torch.float64):
    if int(order) != order or order < 1:
        raise ValueError("order is positive nodes per axis")
    k = torch.arange(order, device=device, dtype=dtype)
    return torch.cos((2*k+1)*math.pi/(2*order)).flip(0)


def lagrange_basis(points, nodes):
    """Polynomial cardinal basis, stable at exact nodes (no 0/0 branch)."""
    values=[]
    for a in range(nodes.numel()):
        out=torch.ones_like(points)
        for b in range(nodes.numel()):
            if b != a:
                out=out*(points-nodes[b])/(nodes[a]-nodes[b])
        values.append(out)
    return torch.stack(values, dim=-1)


def tensor_basis(points, nodes):
    bx=lagrange_basis(points[...,0],nodes)
    by=lagrange_basis(points[...,1],nodes)
    return (by[..., :, None]*bx[..., None, :]).flatten(-2)


def tensor_nodes(order, device=None, dtype=torch.float64):
    n=chebyshev_nodes(order,device,dtype)
    yy,xx=torch.meshgrid(n,n,indexing="ij")
    return torch.stack((xx.flatten(),yy.flatten()),-1)


def transfer_matrices(order, device=None, dtype=torch.float64):
    """T[q,a,b]=parent basis a at child q's node b; q=2*ybit+xbit."""
    n=chebyshev_nodes(order,device,dtype)
    xy=tensor_nodes(order,device,dtype)
    matrices=[]
    for q in range(4):
        center=xy.new_tensor([-.5+(q%2),-.5+(q//2)])
        matrices.append(tensor_basis(xy*.5+center,n).transpose(0,1))
    return torch.stack(matrices)


@dataclass(frozen=True)
class Geometry:
    grid: tuple
    order: int
    leaf_size: int
    depth: int
    side: int
    coordinates: torch.Tensor
    point_leaf: torch.Tensor
    point_basis: torch.Tensor
    transfer: torch.Tensor
    parents: tuple
    quadrants: tuple
    m2l_sources: tuple
    m2l_targets: tuple
    m2l_offsets: tuple
    near_sources: torch.Tensor
    near_targets: torch.Tensor
    cell_nodes: tuple
    cell_sizes: tuple
    stats: dict


def _build_geometry(grid, order, leaf_size, device, dtype):
    rows,cols=grid
    if leaf_size < 1 or int(leaf_size) != leaf_size:
        raise ValueError("leaf_size must be a positive integer, in patches per side")
    target_side=max(math.ceil(rows/leaf_size),math.ceil(cols/leaf_size))
    side=1 << (target_side-1).bit_length()
    depth=side.bit_length()-1
    coords=patch_coordinates(grid,device,dtype)
    rr=torch.arange(rows,device=device).repeat_interleave(cols)
    cc=torch.arange(cols,device=device).repeat(rows)
    leaf=(rr//leaf_size)*side+cc//leaf_size
    local=torch.stack((2*((cc%leaf_size).to(dtype)+.5)/leaf_size-1,2*((rr%leaf_size).to(dtype)+.5)/leaf_size-1),-1)
    basis=tensor_basis(local,chebyshev_nodes(order,device,dtype))
    parents=[torch.empty(0,device=device,dtype=torch.long)]
    quadrants=[torch.empty(0,device=device,dtype=torch.long)]
    sources=[]; targets=[]; offsets=[]; node_levels=[]; sizes=[]
    nodes=tensor_nodes(order,device,dtype)
    leaf_members=[[] for _ in range(side*side)]
    for idx in range(rows*cols):
        leaf_members[(idx//cols//leaf_size)*side+(idx%cols//leaf_size)].append(idx)
    near_s=[]; near_t=[]
    for ty in range(side):
        for tx in range(side):
            receivers=leaf_members[ty*side+tx]
            if not receivers: continue
            for sy in range(max(0,ty-1),min(side,ty+2)):
                for sx in range(max(0,tx-1),min(side,tx+2)):
                    for i in receivers:
                        for j in leaf_members[sy*side+sx]:
                            near_t.append(i); near_s.append(j)
    for level in range(depth+1):
        s=2**level
        cell_width=leaf_size*side/s/cols
        cell_height=leaf_size*side/s/rows
        sizes.append((cell_width,cell_height))
        cy,cx=torch.meshgrid(torch.arange(s,device=device),torch.arange(s,device=device),indexing='ij')
        centers=torch.stack(((cx.flatten().to(dtype)+.5)*cell_width,(cy.flatten().to(dtype)+.5)*cell_height),-1)
        node_levels.append(centers[:,None,:]+nodes[None,:,:]*coords.new_tensor([cell_width,cell_height])/2)
        if level:
            parents.append((cy.flatten()//2)*(s//2)+cx.flatten()//2)
            quadrants.append((cy.flatten()%2)*2+cx.flatten()%2)
        src=[]; dst=[]; off=[]
        if level >= 2:
            for ty in range(s):
                for tx in range(s):
                    for py in range(max(0,ty//2-1),min(s//2,ty//2+2)):
                        for px in range(max(0,tx//2-1),min(s//2,tx//2+2)):
                            for dy in range(2):
                                for dx in range(2):
                                    sy,sx=2*py+dy,2*px+dx
                                    if max(abs(sy-ty),abs(sx-tx)) > 1:
                                        src.append(sy*s+sx); dst.append(ty*s+tx)
                                        off.append((sy-ty+3)*7+sx-tx+3)
        sources.append(torch.tensor(src,device=device,dtype=torch.long))
        targets.append(torch.tensor(dst,device=device,dtype=torch.long))
        offsets.append(torch.tensor(off,device=device,dtype=torch.long))
    cells=sum(4**l for l in range(depth+1))
    n=rows*cols; r=order**2
    stats={'grid_positions':n,'allocated_leaf_cells':side*side,'occupied_leaf_cells':sum(bool(v) for v in leaf_members),'allocated_cells':cells,'depth':depth,'coefficients_per_cell':r,'near_ordered_pairs':len(near_s),'near_degree_bound':9*leaf_size**2,'m2l_cell_pairs':sum(len(v) for v in sources),'m2l_max_degree':max([0]+[int(torch.bincount(v,minlength=4**l).max()) if len(v) else 0 for l,v in enumerate(targets)]),'production_dense_token_pairs':0,'m2l_scalar_work_per_feature':sum(len(v)*r*r for v in sources),'transfer_scalar_work_per_feature':2*(cells-1)*r*r}
    return Geometry(grid,order,leaf_size,depth,side,coords,leaf,basis,transfer_matrices(order,device,dtype),tuple(parents),tuple(quadrants),tuple(sources),tuple(targets),tuple(offsets),torch.tensor(near_s,device=device,dtype=torch.long),torch.tensor(near_t,device=device,dtype=torch.long),tuple(node_levels),tuple(sizes),stats)


@lru_cache(maxsize=24)
def _cached_geometry(grid,order,leaf_size,device,dtype):
    return _build_geometry(grid,order,leaf_size,torch.device(device),dtype)


def build_geometry(grid, order=2, leaf_size=2, device='cpu', dtype=torch.float32):
    """Cache contains geometric constants only; visibility never enters this cache."""
    if len(grid)!=2 or any(int(v)!=v or v<1 for v in grid):
        raise ValueError("grid must contain two positive integers")
    if int(order)!=order or order<1 or int(leaf_size)!=leaf_size or leaf_size<1:
        raise ValueError("order and leaf_size must be positive integers")
    grid=tuple(map(int,grid))
    return _cached_geometry(grid,int(order),int(leaf_size),str(torch.device(device)),dtype)


def clear_geometry_cache():
    _cached_geometry.cache_clear()


def ownership_table(geometry, include_self=True):
    """EXHAUSTIVE TEST ONLY: NxN count of distinct near/far owners."""
    n=math.prod(geometry.grid)
    if n>256:
        raise ValueError("dense pair ownership is restricted to tiny correctness fixtures")
    counts=torch.zeros(n,n,dtype=torch.int64)
    counts[geometry.near_targets.cpu(),geometry.near_sources.cpu()]+=1
    leaf=geometry.point_leaf.cpu()
    ly=leaf//geometry.side; lx=leaf%geometry.side
    for level in range(2,geometry.depth+1):
        shift=geometry.depth-level; side=2**level
        cells=(ly//(2**shift))*side+lx//(2**shift)
        for target,source in zip(geometry.m2l_targets[level].cpu().tolist(),geometry.m2l_sources[level].cpu().tolist()):
            receiver=(cells==target).nonzero().flatten()
            sender=(cells==source).nonzero().flatten()
            if receiver.numel() and sender.numel():
                counts[receiver[:,None],sender[None,:]]+=1
    if not include_self:
        counts.fill_diagonal_(0)
    return counts


def pair_owner(receiver_idx,source_idx,grid,leaf_size=2):
    """Return 0 for direct near ownership, else unique far interaction level.

    No pair table is allocated. Useful for selected-query teaching diagrams.
    """
    rows,cols=map(int,grid); n=rows*cols
    if not (0<=receiver_idx<n and 0<=source_idx<n):
        raise ValueError("point index outside grid")
    target_side=max(math.ceil(rows/leaf_size),math.ceil(cols/leaf_size))
    side=1 << (target_side-1).bit_length(); depth=side.bit_length()-1
    ry,rx=receiver_idx//cols//leaf_size,receiver_idx%cols//leaf_size
    sy,sx=source_idx//cols//leaf_size,source_idx%cols//leaf_size
    for level in range(2,depth+1):
        divisor=2**(depth-level)
        if max(abs(ry//divisor-sy//divisor),abs(rx//divisor-sx//divisor))>1:
            return level
    return 0
